In [ ]:
# =============================================================================
# jobs/36_ingest_rfg.py
# Rapid Flood Guidance (RFG) ingestion, from the Flood Forecasting Centre API (the same API and key as job 01).
#
# What this does:
#   1. Gets the latest RFG update (v3 /rfg/latest). If no guidance is current the API returns only last_updated and the status
#      "No Rapid Flood Guidance at this time"; that is stored too, as a row of its own, so the history shows when guidance started and stopped.
#   2. If that version is already stored (same key and same last_updated) it exits. Otherwise it replaces the update's rows in:
#        config.TBL_RFG_POLYGONS  one row per polygon per update (GeoJSON, latest update text, counties, is_valid)
#        config.TBL_RFG_UPDATES   one row per update, written LAST, so its presence means the polygons are complete
#   3. The first time it runs (no table yet), or with backfill = yes, it also loads the 50 most recent updates from /rfg, so there is history.
#   4. Sets the task value new_data, so a workflow can run job 37 (what lies under each polygon) only when something was stored.
#
# RFG follows rapid flooding (heavy showers), so it can be issued at any hour: there is no polling window here.
#
# HEARTBEAT  Every run, even when nothing changed, overwrites one row of config.TBL_RFG_CHECKS with when it checked and what the feed said. The updates
# table only grows when guidance changes, so without this a quiet feed cannot be told from a poll that has stopped.
#
# Exit values: "success" (stored), "no_change", "no_data".
# Widgets: backfill (auto, yes, no), backfill_pages.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import importlib

import pandas as pd
from pyspark.sql.types import BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
from utils.rfg import update_key as _update_key

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.rfg as _rfg
importlib.reload(_rfg)
from utils.helpers import ffc_get, get_spark, table_exists, utc_now
from utils.rfg import POLYGON_COLS, UPDATE_COLS, is_stored, parse_update

spark = get_spark()
now = utc_now()

dbutils.widgets.dropdown("backfill", "auto", ["auto", "yes", "no"], "Load the recent history (auto = only when there is no table yet)")
dbutils.widgets.text("backfill_pages", "1", "Pages of 50 updates to load when backfilling")
BACKFILL = dbutils.widgets.get("backfill")
PAGES = max(1, int(dbutils.widgets.get("backfill_pages") or 1))


def set_new_data(flag):
    try:
        dbutils.jobs.taskValues.set(key="new_data", value="true" if flag else "false")
    except Exception:
        pass   # not running as a job task (interactive run)


set_new_data(False)
API_KEY = config.get_ffc_api_key()


def record_check(result, latest=None):
    """The heartbeat: one row, overwritten, saying when the feed was last checked, how the run ended and what the latest update was."""
    try:
        row = [(now.replace(tzinfo=None), result, _update_key(latest) if isinstance(latest, dict) and latest else None,
                str(latest.get("last_updated")) if isinstance(latest, dict) and latest else None, str(latest.get("status")) if isinstance(latest, dict) and latest else None)]
        schema = StructType([StructField("checked_at", TimestampType()), StructField("result", StringType()), StructField("latest_update_key", StringType()),
                             StructField("latest_last_updated", StringType()), StructField("latest_status", StringType())])
        spark.createDataFrame(row, schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(config.TBL_RFG_CHECKS)
    except Exception as e:                      # a heartbeat that cannot be written must never stop the ingest
        print(f"Could not record the check: {str(e)[:150]}")

In [ ]:
# =============================================================================
# 1. THE LATEST UPDATE, AND WHETHER IT IS ALREADY STORED
# =============================================================================

latest = ffc_get(f"{config.FFC_BASE_URL}{config.FFC_RFG_LATEST_PATH}", API_KEY)
if not isinstance(latest, dict) or not latest:
    print("The API returned nothing for the latest RFG. Exiting.")
    record_check("no_data")
    dbutils.notebook.exit("no_data")
if "status" not in latest and "id" not in latest:
    raise Exception(f"Unexpected RFG response shape. Keys: {sorted(latest.keys())}")
print(f"Latest RFG: id={latest.get('id')} status={latest.get('status')!r} last_updated={latest.get('last_updated')}")

exists = table_exists(spark, config.TBL_RFG_UPDATES)
stored = spark.table(config.TBL_RFG_UPDATES).select("update_key", "last_updated").toPandas() if exists else None
do_backfill = BACKFILL == "yes" or (BACKFILL == "auto" and not exists)

if is_stored(stored, latest) and not do_backfill:
    print("No new or modified RFG since the last poll. Exiting.")
    record_check("no_change", latest)
    dbutils.notebook.exit("no_change")

In [ ]:
# =============================================================================
# 2. COLLECT THE UPDATES TO STORE: the latest, and the recent history when backfilling
# =============================================================================

updates = []
if not is_stored(stored, latest):
    updates.append(latest)
if do_backfill:
    for page in range(1, PAGES + 1):
        listing = ffc_get(f"{config.FFC_BASE_URL}{config.FFC_RFG_PATH}", API_KEY, params={"page_number": page, "page_size": 50}, timeout=120)
        if "rfg_updates" not in listing:
            raise Exception(f"Unexpected RFG list shape. Keys: {sorted(listing.keys())}")
        batch = listing["rfg_updates"]
        print(f"History page {page}: {len(batch)} updates.")
        if not batch:
            break
        updates += [u for u in batch if not is_stored(stored, u)]
seen, unique = set(), []
for u in updates:                       # the latest can also be in the list: keep one copy of each version
    k = (str(u.get("id")), str(u.get("last_updated")))
    if k not in seen:
        seen.add(k)
        unique.append(u)
updates = unique
print(f"{len(updates)} updates to store.")
if not updates:
    record_check("no_change", latest)
    dbutils.notebook.exit("no_change")

ingested_at = pd.Timestamp(now)
urows, prows = [], []
for u in updates:
    ur, pr = parse_update(u, ingested_at)
    urows.append(ur)
    prows += pr
upd, pol = pd.DataFrame(urows, columns=UPDATE_COLS), pd.DataFrame(prows, columns=POLYGON_COLS)
print(f"Polygons: {len(pol)} in total; without usable coordinates: {int(pol['geometry'].isna().sum()) if len(pol) else 0}; marked not valid: {int((~pol['is_valid']).sum()) if len(pol) else 0}.")
display(upd[["update_key", "status", "is_present", "issued_at", "last_updated", "n_polygons", "n_valid"]].head(20).astype(str))

In [ ]:
# =============================================================================
# 3. WRITE: the polygons first, the update rows last
# A re-run replaces the rows of the same updates, so it never duplicates, and a failure part way leaves the update looking not stored,
# so the next poll repeats it.
# =============================================================================

UPDATE_SCHEMA = StructType([
    StructField("update_key", StringType()), StructField("rfg_id", LongType()), StructField("status", StringType()), StructField("is_present", BooleanType()),
    StructField("issued_at", TimestampType()), StructField("last_updated", TimestampType()), StructField("date_issued", StringType()),
    StructField("time_issued", TimestampType()), StructField("pdf_url", StringType()), StructField("national_summary", StringType()),
    StructField("n_polygons", IntegerType()), StructField("n_valid", IntegerType()), StructField("ingested_at", TimestampType())])
POLYGON_SCHEMA = StructType([
    StructField("update_key", StringType()), StructField("rfg_id", LongType()), StructField("polygon_number", IntegerType()), StructField("geometry", StringType()),
    StructField("updated_at", TimestampType()), StructField("latest_update", StringType()), StructField("counties", StringType()),
    StructField("is_valid", BooleanType()), StructField("ingested_at", TimestampType())])


def to_rows(df, schema):
    out = []
    for r in df.itertuples(index=False):
        row = []
        for f, v in zip(schema.fields, r):
            if v is None or (not isinstance(v, (list, tuple)) and pd.isna(v)):
                row.append(None)
            elif isinstance(f.dataType, TimestampType):
                row.append(pd.Timestamp(v).to_pydatetime())
            elif isinstance(f.dataType, (LongType, IntegerType)):
                row.append(int(v))
            elif isinstance(f.dataType, BooleanType):
                row.append(bool(v))
            else:
                row.append(str(v))
        out.append(tuple(row))
    return out


def replace_rows(table, keys, df, schema):
    if table_exists(spark, table):
        in_list = ", ".join("'" + k.replace("'", "''") + "'" for k in keys)
        spark.sql(f"DELETE FROM {table} WHERE update_key IN ({in_list})")
    if len(df):
        spark.createDataFrame(to_rows(df, schema), schema).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)


keys = list(upd["update_key"])
replace_rows(config.TBL_RFG_POLYGONS, keys, pol, POLYGON_SCHEMA)
replace_rows(config.TBL_RFG_UPDATES, keys, upd, UPDATE_SCHEMA)
spark.sql(f"COMMENT ON TABLE {config.TBL_RFG_UPDATES} IS 'Rapid Flood Guidance updates from the Flood Forecasting Centre API, one row per update and one per no-guidance state (update_key none_<time>). is_present says guidance was in force. Public. Job 36.'")
spark.sql(f"COMMENT ON TABLE {config.TBL_RFG_POLYGONS} IS 'The polygons of each Rapid Flood Guidance update: GeoJSON (WGS84), the latest update text, counties and whether the polygon is valid. Public. Job 36.'")
print(f"Stored {len(upd)} updates and {len(pol)} polygons.")
set_new_data(bool(upd["is_present"].any()))
record_check("success", latest)

In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")